# Act 158 Pathways — Dataset Documentation

**Dataset:** `act158_pathways.csv`

**Purpose:** Tracks proposed Act 158 graduation pathways and supporting evidence statuses for students within the synthetic Graduation Tracker student information system.

**Record level:** One row per student.

**Primary key:** `pathway_record_id`

**Foreign key:** `student_id` → `students.csv`

**Fields:**

- `pathway_record_id`: Unique pathway record identifier using the format `PTH000001`.
- `student_id`: Student identifier linking pathway records to the student dataset.
- `proposed_pathway`: Assigned graduation pathway (Pathway 1, Pathway 2, Pathway 3, Pathway 4, or Pathway 5).
- `evidence_status`: Status of supporting pathway documentation (Verified, Pending Review, Missing Evidence, or Not Yet Evaluated).
- `evidence_reference`: Unique evidence reference identifier using the format `EV000001`. Null when evidence is missing or has not yet been evaluated.

**Business rules:**

- Every student receives one proposed Act 158 graduation pathway.
- Pathways 1–5 are assigned randomly with equal probability.
- Evidence statuses follow these probability distributions:
  - Verified: 35%
  - Pending Review: 25%
  - Missing Evidence: 25%
  - Not Yet Evaluated: 15%
- Evidence references must be null when the status is Missing Evidence or Not Yet Evaluated.
- Verified and Pending Review records receive an evidence reference.
- Pathway record IDs must be unique.
- Every `student_id` must exist in the student dataset.
- Each student must have exactly one pathway record.
- Proposed pathway assignments do not independently establish graduation eligibility.

**Data source:** Synthetic Act 158 pathway records generated using the student roster and NumPy random sampling with a fixed random seed of 42.

**Limitations:** Pathway assignments and evidence statuses are randomly generated and do not reflect actual Keystone assessment results, student eligibility, or verified completion of Pennsylvania Act 158 graduation requirements. Evidence references are synthetic identifiers rather than actual documentation. The dataset represents a simplified pathway tracking system for portfolio demonstration and does not model all eligibility criteria associated with Pennsylvania's five graduation pathways.

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

In [ ]:
# Connect Google Drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Create folder path

folder_path = Path('/content/drive/MyDrive/Graduation_Tracker/data/ground_truth')

In [ ]:
# Set random seed

rng = np.random.default_rng(42)

In [ ]:
# Generate Pathway Completion dataset

students = pd.read_csv(folder_path/'students.csv')

pathways = ['Pathway 1', 'Pathway 2', 'Pathway 3', 'Pathway 4', 'Pathway 5']

status = ['Verified', 'Pending Review', 'Missing Evidence', 'Not Yet Evaluated']

pathway = pd.DataFrame({
    'student_id':students.student_id,
    'proposed_pathway':rng.choice(pathways, len(students)),
    'evidence_status':rng.choice(status, len(students), p = [0.35, 0.25, 0.25, 0.15]),
    'evidence_reference':[f'EV{i:06d}' for i in range(1, len(students) + 1)]
})

pathway.loc[pathway.evidence_status.isin(['Missing Evidence', 'Not Yet Evaluated']), 'evidence_reference'] = pd.NA

pathway.insert(0, 'pathway_record_id', [f'PTH{i:06d}' for i in range(1, len(pathway) + 1)])

display(pathway.head())

,pathway_record_id,student_id,proposed_pathway,evidence_status,evidence_reference
0,PTH000001,1145269,Pathway 1,Verified,EV000001
1,PTH000002,1145270,Pathway 4,Missing Evidence,<NA>
2,PTH000003,1145271,Pathway 4,Not Yet Evaluated,<NA>
3,PTH000004,1145272,Pathway 3,Missing Evidence,<NA>
4,PTH000005,1145273,Pathway 3,Verified,EV000005


In [ ]:
# Validate dataset

assert pathway.pathway_record_id.is_unique

assert pathway.student_id.isin(students.student_id).all()

assert pathway.loc[pathway.evidence_status.eq('Missing Evidence'), 'evidence_reference'].isna().all()

print(pathway.evidence_status.value_counts())


evidence_status
Verified             1806
Pending Review       1245
Missing Evidence     1237
Not Yet Evaluated     712
Name: count, dtype: int64


In [ ]:
# Save CSV to Google Drive

csv_path = folder_path / 'act158_pathways.csv'

pathway.to_csv(csv_path, index = False)